# MetrôBot SP 2.0

**Disciplina:** Inteligência Artificial e Machine Learning  
**Integrantes:**  
- Pedro Paulo Camargo da Silva

## Como executar

1. Execute o notebook do início ao fim usando **Executar tudo**.
2. O projeto pode ser executado com `PROVEDOR = "offline"` sem necessidade de chave de API.
3. Para utilizar o Groq, configure `GROQ_API_KEY` nos Secrets do Google Colab.
4. Ao final, execute `rodar_testes()` para validar os testes do projeto.

In [16]:
%pip install -q groq ollama ipywidgets python-dotenv

In [17]:
# ---------- CONFIGURAÇÃO

import os, json, re, unicodedata
from collections import deque
from itertools import product

In [18]:
# ---------- LLM

# Provedores disponíveis: "groq", "ollama" ou "offline"
PROVEDOR = "groq"

# Modelo utilizado via Groq
MODELO_GROQ = "openai/gpt-oss-20b"

# Modelo utilizado caso o provedor seja Ollama
MODELO_OLLAMA = "llama3.2"

def obter_chave_groq():
    """Busca a chave SEM escrevê-la no código: Colab Secrets → .env → variável de ambiente."""
    try:
        from google.colab import userdata
        return userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass
    return os.environ.get("GROQ_API_KEY")

def chamar_llm(mensagens, modo_json=False):
    """Envia mensagens ao modelo configurado e devolve o texto da resposta."""
    if PROVEDOR == "groq":
        from groq import Groq
        cliente = Groq(api_key=obter_chave_groq())
        extras = {"response_format": {"type": "json_object"}} if modo_json else {}
        resposta = cliente.chat.completions.create(
            model=MODELO_GROQ, messages=mensagens, temperature=0, **extras)
        return resposta.choices[0].message.content
    elif PROVEDOR == "ollama":
        import ollama
        extras = {"format": "json"} if modo_json else {}
        resposta = ollama.chat(model=MODELO_OLLAMA, messages=mensagens,
                               options={"temperature": 0}, **extras)
        return resposta["message"]["content"]
    else:
        raise RuntimeError("Modo offline: nenhum LLM configurado.")

In [19]:
# --- GRAFO -----

LINHAS = {
    "Linha 1-Azul": [
        "Tucuruvi", "Parada Inglesa", "Jardim São Paulo", "Santana",
        "Carandiru", "Portuguesa-Tietê", "Armênia", "Tiradentes", "Luz",
        "São Bento", "Sé", "Japão-Liberdade", "São Joaquim", "Vergueiro",
        "Paraíso", "Ana Rosa", "Vila Mariana", "Santa Cruz",
        "Praça da Árvore", "Saúde", "São Judas", "Conceição", "Jabaquara",
    ],
    "Linha 2-Verde": [
        "Vila Madalena", "Sumaré", "Clínicas", "Consolação", "Trianon-Masp",
        "Brigadeiro", "Paraíso", "Ana Rosa", "Chácara Klabin",
        "Santos-Imigrantes", "Alto do Ipiranga", "Sacomã", "Tamanduateí",
        "Vila Prudente",
    ],
    "Linha 3-Vermelha": [
        "Palmeiras-Barra Funda", "Marechal Deodoro", "Santa Cecília",
        "República", "Anhangabaú", "Sé", "Pedro II", "Brás",
        "Bresser-Mooca", "Belém", "Tatuapé", "Carrão", "Penha",
        "Vila Matilde", "Guilhermina-Esperança", "Patriarca-Vila Ré",
        "Artur Alvim", "Corinthians-Itaquera",
    ],
}

CORES = {
         "Linha 1-Azul": "#1e88e5",
         "Linha 2-Verde": "#2e7d32",
         "Linha 3-Vermelha": "#d32f2f"}

# ----- Constrói o grafo e registra a linha de cada trecho ----------

def construir_grafo_multilinhas(linhas):
    """Retorna (grafo, linhas_do_trecho).

    grafo: {estacao: [vizinhas sem repetição]}
    linhas_do_trecho: {(a, b): {nomes das linhas}} — guarda (a,b) e (b,a)
    """

    grafo = {}
    linhas_do_trecho = {}

    for nome_linha, estacoes in linhas.items():
        for i in range(len(estacoes) - 1):
            a, b = estacoes[i], estacoes[i + 1]

            # TODO 1: garantir que a e b existem no grafo
            if a not in grafo:
                grafo[a] = []

            if b not in grafo:
                grafo[b] = []

            # TODO 2: adicionar vizinhos sem duplicar
            if b not in grafo[a]:
                grafo[a].append(b)

            if a not in grafo[b]:
                grafo[b].append(a)

            # TODO 3: registrar a linha nas duas direções
            if (a, b) not in linhas_do_trecho:
                linhas_do_trecho[(a, b)] = set()

            if (b, a) not in linhas_do_trecho:
                linhas_do_trecho[(b, a)] = set()

            linhas_do_trecho[(a, b)].add(nome_linha)
            linhas_do_trecho[(b, a)].add(nome_linha)

    return grafo, linhas_do_trecho


GRAFO, LINHAS_DO_TRECHO = construir_grafo_multilinhas(LINHAS)

In [20]:
# ---------- LOCAIS ----------
LOCAIS = {
    # Linha 1 - Azul
    "Shopping Metrô Tucuruvi": "Tucuruvi",
    "Terminal Rodoviário Tietê": "Portuguesa-Tietê",
    "Museu de Arte Sacra": "Tiradentes",
    "Pinacoteca": "Luz",
    "Museu da Língua Portuguesa": "Luz",
    "Mosteiro de São Bento": "São Bento",
    "Rua 25 de Março": "São Bento",
    "Catedral da Sé": "Sé",
    "Bairro da Liberdade": "Japão-Liberdade",
    "Centro Cultural São Paulo": "Vergueiro",
    "Shopping Metrô Santa Cruz": "Santa Cruz",
    "Universidade São Judas": "São Judas",
    "Terminal Rodoviário Jabaquara": "Jabaquara",

    # Linha 2-Verde
    "MASP": "Trianon-Masp",
    "Hospital das Clínicas": "Clínicas",
    "Shopping Center 3": "Consolação",

    # Linha 3-Vermelha
    "Galeria do Rock": "República",
    "Theatro Municipal de São Paulo": "Anhangabaú",
    "Neo Química Arena": "Corinthians-Itaquera",

}


In [21]:
# ---------- BFS ----------
def reconstruir_caminho(pai, destino):

    caminho = []
    atual = destino
    while atual is not None:
        caminho.append(atual)
        atual = pai[atual]
    return list(reversed(caminho))

def bfs(grafo, origem, destino, bloqueadas=()):
    if origem in bloqueadas or destino in bloqueadas:   #Se a origem estiver nas bloqueadas OU o destino estiver nas bloqueadas, encerre.
        return None, []
    fila = deque([origem])
    pai = {origem: None}
    ordem_visita = []
    while fila:
        atual = fila.popleft()          # 1º da fila sai primeiro (FIFO)
        ordem_visita.append(atual)
        if atual == destino:
            return reconstruir_caminho(pai, destino), ordem_visita
        for vizinho in grafo[atual]:
            if vizinho not in pai and vizinho not in bloqueadas:
                pai[vizinho] = atual
                fila.append(vizinho)
    return None, ordem_visita


In [22]:
# ---------- DFS ----------
def dfs(grafo, origem, destino, bloqueadas=()):
    """Vai fundo no 1º vizinho; se não achar, volta (backtracking) e tenta o próximo."""
    if origem in bloqueadas or destino in bloqueadas:
        return None, []

    visitados = set()
    ordem_visita = []

    def explorar(atual, caminho):
        visitados.add(atual)
        ordem_visita.append(atual)

        if atual == destino:
            return caminho

        for vizinho in grafo[atual]:
            if vizinho not in visitados and vizinho not in bloqueadas:
                resultado = explorar(vizinho, caminho + [vizinho])   # mergulha

                if resultado:
                    return resultado
        return None                     # beco sem saída → volta

    return explorar(origem, [origem]), ordem_visita

In [23]:
# ---------- BALDEAÇÕES ----------

def contar_baldeacoes(caminho, linhas_do_trecho):
    """Retorna (quantidade, lista de (estacao, linha_nova))."""

    if not caminho or len(caminho) < 2:
        return 0, []

    baldeacoes = []

    # Linhas disponíveis no primeiro trecho
    linhas_atuais = linhas_do_trecho[(caminho[0], caminho[1])]
    linha_atual = next(iter(linhas_atuais))

    # Percorre os próximos trechos
    for i in range(1, len(caminho) - 1):
        a = caminho[i]
        b = caminho[i + 1]

        linhas_proximo_trecho = linhas_do_trecho[(a, b)]

        # Continua na mesma linha sempre que possível
        if linha_atual in linhas_proximo_trecho:
            continue

        # A linha atual não atende ao próximo trecho:
        # é necessário fazer uma baldeação em "a"
        linha_atual = next(iter(linhas_proximo_trecho))
        baldeacoes.append((a, linha_atual))

    return len(baldeacoes), baldeacoes

In [24]:
def pode_embarcar(P, Q, R):
    """P: estação aberta | Q: precisa de acessibilidade | R: elevador funcionando"""
    return P and ((not Q) or R)

def tabela_verdade():
    print(" P     | Q     | R     | P ∧ (¬Q ∨ R)")
    print("-" * 40)
    for P, Q, R in product([True, False], repeat=3):
        print(f" {P!s:5} | {Q!s:5} | {R!s:5} | {pode_embarcar(P, Q, R)}")

tabela_verdade()



 P     | Q     | R     | P ∧ (¬Q ∨ R)
----------------------------------------
 True  | True  | True  | True
 True  | True  | False | False
 True  | False | True  | True
 True  | False | False | True
 False | True  | True  | False
 False | True  | False | False
 False | False | True  | False
 False | False | False | False


In [25]:
def fatos_base():
    """Cria os fatos fixos das estações, linhas e locais conhecidos."""
    fatos = set()

    for nome_linha, estacoes in LINHAS.items():
        for estacao in estacoes:
            fatos.add(("estacao", estacao))
            fatos.add(("pertence", estacao, nome_linha))

    for local, estacao in LOCAIS.items():
        fatos.add(("proximo_de", local, estacao))

    return fatos

def consultar(fatos, predicado):
    """Devolve os argumentos de todos os fatos de um predicado."""
    return [f[1:] for f in fatos if f[0] == predicado]


def r_origem(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_esta_em"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("origem", e))
    for (e,) in consultar(fatos, "usuario_esta_na_estacao"):
        novos.add(("origem", e))
    return novos

def r_destino(fatos):
    novos = set()
    for (local,) in consultar(fatos, "usuario_quer_ir"):
        for (l, e) in consultar(fatos, "proximo_de"):
            if l == local:
                novos.add(("destino", e))
    for (e,) in consultar(fatos, "usuario_quer_ir_estacao"):
        novos.add(("destino", e))
    return novos

def r_bloqueio(fatos):
    return {("bloqueada", e) for (e,) in consultar(fatos, "fechada")}

def r_acessibilidade(fatos):
    if not consultar(fatos, "precisa_acessibilidade"):
        return set()
    return {("inacessivel", e) for (e,) in consultar(fatos, "elevador_em_manutencao")}

def r_alerta(fatos):
    novos = set()
    inacessiveis = {e for (e,) in consultar(fatos, "inacessivel")}
    for papel in ("origem", "destino"):
        for (e,) in consultar(fatos, papel):
            if e in inacessiveis:
                novos.add(("alerta", papel, e))
    return novos

def r_integracao(fatos):
    novos = set()

    pertencimentos = consultar(fatos, "pertence")

    for e1, l1 in pertencimentos:
        for e2, l2 in pertencimentos:
            if e1 == e2 and l1 != l2:
                novos.add(("integracao", e1))

    return novos

# Regra Escolhida
def r_linha_paralisada(fatos):
    novos = set()

    linhas_paralisadas = {
        linha for (linha,) in consultar(fatos, "linha_paralisada")
    }

    for estacao, linha in consultar(fatos, "pertence"):
        if linha in linhas_paralisadas:
            novos.add(("bloqueada", estacao))

    return novos


REGRAS = [
    ("R1 origem",
     "∀l ∀e (usuario_esta_em(l) ∧ proximo_de(l,e) → origem(e))",
     r_origem),

    ("R2 destino",
     "∀l ∀e (usuario_quer_ir(l) ∧ proximo_de(l,e) → destino(e))",
     r_destino),

    ("R3 bloqueio",
     "∀e (fechada(e) → bloqueada(e))",
     r_bloqueio),

    ("R4 acessibilidade",
     "∀e (precisa_acessibilidade ∧ elevador_em_manutencao(e) → inacessivel(e))",
     r_acessibilidade),

    ("R5 alerta",
     "∀p ∀e (papel(p,e) ∧ inacessivel(e) → alerta(p,e))",
     r_alerta),

    ("R6 integração",
     "∀e ∀l1 ∀l2 (pertence(e,l1) ∧ pertence(e,l2) ∧ l1 ≠ l2 → integracao(e))",
     r_integracao),

    ("R7 linha paralisada",
     "∀e ∀l (linha_paralisada(l) ∧ pertence(e,l) → bloqueada(e))",
     r_linha_paralisada),
]


def encadear_para_frente(fatos, regras, verbose=False):
    """Aplica as regras em rodadas até não surgir nenhum fato novo (ponto fixo)."""
    fatos = set(fatos)
    justificativas = {}
    rodada = 0
    while True:
        rodada += 1
        novos_na_rodada = set()
        for nome, _formula, regra in regras:
            for fato in regra(fatos) - fatos:
                novos_na_rodada.add(fato)
                justificativas[fato] = nome
        if verbose:
            print(f"Rodada {rodada}: {len(novos_na_rodada)} fato(s) novo(s)")
        if not novos_na_rodada:
            return fatos, justificativas
        fatos |= novos_na_rodada

In [26]:
# ---------- INTÉRPRETE ----------

# Lista única com todas as estações das 3 linhas
TODAS_ESTACOES = list(GRAFO.keys())


def normalizar(texto):
    """Minúsculas e sem acentos: 'São Bento' → 'sao bento'."""
    texto = unicodedata.normalize("NFD", texto.lower())
    return "".join(
        c for c in texto
        if unicodedata.category(c) != "Mn"
    )


def resolver_nome(nome):
    """GUARDRAIL: só aceita estações e locais que existem de verdade."""
    if not nome:
        return None

    alvo = normalizar(nome).strip()

    # Procura nas 52 estações
    for estacao in TODAS_ESTACOES:
        if normalizar(estacao) == alvo:
            return ("estacao", estacao)

    # Procura nos locais conhecidos
    for local in LOCAIS:
        if normalizar(local) == alvo:
            return ("local", local)

    return None


PROMPT_INTERPRETE = """Você é o módulo de INTERPRETAÇÃO do MetrôBot SP 2.0.
Sua única tarefa é transformar o pedido do passageiro em JSON.

Estações válidas:
{estacoes}

Locais conhecidos:
{locais}

Responda APENAS com um JSON neste formato:
{{"origem": "<nome exato de estação ou local, ou null>",
  "destino": "<nome exato de estação ou local, ou null>",
  "acessibilidade": <true ou false>}}

Regras:
- Use SOMENTE nomes presentes nas listas acima.
- Preserve exatamente o nome apresentado na lista.
- "acessibilidade" deve ser true se o passageiro mencionar cadeira de rodas,
  mobilidade reduzida, muletas, carrinho de bebê ou necessidade de elevador.
- Se não conseguir identificar origem ou destino, use null.
- Nunca invente estações ou locais.
"""


def interpretar_offline(texto):
    """
    Plano B sem LLM:
    procura estações e locais conhecidos diretamente no texto.
    """

    texto_min = texto.lower()
    texto_sem = normalizar(texto)

    # Agora considera as estações das 3 linhas
    candidatos = (
        [(n, "estacao") for n in TODAS_ESTACOES]
        + [(n, "local") for n in LOCAIS]
    )

    # Nomes maiores primeiro para evitar conflitos
    candidatos.sort(
        key=lambda c: len(c[0]),
        reverse=True
    )

    ocupado = [False] * len(texto_min)
    encontrados = []

    for nome, tipo in candidatos:

        buscas = [
            (texto_min, nome.lower())
        ]

        # Também permite procurar ignorando acentos
        if len(nome) > 4 and len(texto_sem) == len(texto_min):
            buscas.append(
                (texto_sem, normalizar(nome))
            )

        for base, padrao in buscas:

            for m in re.finditer(
                r"(?<!\w)" + re.escape(padrao) + r"(?!\w)",
                base
            ):

                if not any(ocupado[m.start():m.end()]):

                    encontrados.append(
                        (m.start(), nome)
                    )

                    for i in range(m.start(), m.end()):
                        ocupado[i] = True

    # Organiza pela posição em que apareceram no texto
    encontrados.sort()

    palavras_acess = [
        "cadeira de rodas",
        "acessibilidade",
        "mobilidade",
        "muleta",
        "carrinho de bebe",
        "elevador"
    ]

    return {
        "origem": (
            encontrados[0][1]
            if len(encontrados) > 0
            else None
        ),

        "destino": (
            encontrados[1][1]
            if len(encontrados) > 1
            else None
        ),

        "acessibilidade": any(
            p in texto_sem
            for p in palavras_acess
        ),
    }


def interpretar_pedido(texto):
    """
    Texto livre → pedido validado.

    Usa o LLM quando disponível.
    Se estiver offline ou ocorrer erro, usa o interpretador offline.
    """

    # ---------- MODO OFFLINE ----------
    if PROVEDOR == "offline":

        bruto = interpretar_offline(texto)
        fonte = "offline"

    # ---------- MODO COM LLM ----------
    else:

        sistema = PROMPT_INTERPRETE.format(
            estacoes=", ".join(TODAS_ESTACOES),
            locais=", ".join(LOCAIS)
        )

        try:

            resposta = chamar_llm(
                [
                    {
                        "role": "system",
                        "content": sistema
                    },
                    {
                        "role": "user",
                        "content": texto
                    }
                ],
                modo_json=True
            )

            bruto = json.loads(resposta)
            fonte = PROVEDOR

        except Exception as erro:

            print(
                f"LLM indisponível ({erro}). "
                "Usando modo offline."
            )

            bruto = interpretar_offline(texto)
            fonte = "offline"

    # ---------- GUARDRAILS ----------

    origem = resolver_nome(
        bruto.get("origem")
    )

    destino = resolver_nome(
        bruto.get("destino")
    )

    if origem is None or destino is None:

        return None, (
            "Não entendi origem/destino "
            f"(resposta bruta: {bruto})"
        )

    # ---------- PEDIDO VALIDADO ----------

    pedido = {
        "origem": origem,
        "destino": destino,
        "acessibilidade": bool(
            bruto.get("acessibilidade")
        )
    }

    return pedido, f"Interpretado via {fonte}"

In [27]:
# ---------- VISUALIZAÇÃO ----------

def desenhar_linhas(resultado):
    """Desenha as 3 linhas do metrô e destaca o resultado da busca."""

    caminho = set(resultado["caminho"] or [])
    visitados = set(resultado["visitados"])
    bloqueadas = set(resultado["bloqueadas"])

    # Estações onde a rota realmente faz baldeação
    estacoes_baldeacao = {
        estacao
        for estacao, _linha in resultado.get("baldeacoes", [])
    }

    linhas_html = []

    # ---------- LEGENDA ----------

    linhas_html.append("""
    <div style="
        font-family:Arial,sans-serif;
        margin-bottom:18px;
        padding:12px;
        background:#292a2d;
        color:#e8eaed;
        border:1px solid #3c4043;
        border-radius:8px;
        font-size:13px;
    ">
        <b>Legenda:</b>
        🔵 Origem/Destino &nbsp;
        🟢 Rota &nbsp;
        🟡 Baldeação &nbsp;
        ⚪ Visitada &nbsp;
        🔴 Bloqueada
    </div>
    """)

    # ---------- PERCORRE AS 3 LINHAS ----------

    for nome_linha, estacoes in LINHAS.items():

        cor_linha = CORES[nome_linha]

        # Título da linha
        linhas_html.append(
            f"""
            <div style="
                margin-top:18px;
                margin-bottom:10px;
                font-family:Arial,sans-serif;
                font-weight:bold;
                font-size:16px;
                color:{cor_linha};
            ">
                {nome_linha}
            </div>
            """
        )

        estacoes_html = []

        # ---------- PERCORRE AS ESTAÇÕES ----------

        for estacao in estacoes:

            # ---------- ESTADO VISUAL DA ESTAÇÃO ----------

            if estacao in bloqueadas:
                cor = "#ef5350"
                marca = "BLOQUEADA"

            elif (
                estacao == resultado["origem"]
                and estacao in caminho
            ):
                cor = "#42a5f5"
                marca = "ORIGEM"

            elif (
                estacao == resultado["destino"]
                and estacao in caminho
            ):
                cor = "#42a5f5"
                marca = "DESTINO"

            elif estacao in estacoes_baldeacao:
                cor = "#fbc02d"
                marca = "BALDEAÇÃO"

            elif estacao in caminho:
                cor = "#4caf50"
                marca = "ROTA"

            elif estacao in visitados:
                cor = "#9e9e9e"
                marca = "VISITADA"

            else:
                cor = "#202124"
                marca = ""

            # ---------- HTML DA ESTAÇÃO ----------

            estacoes_html.append(
                f"""
                <div style="
                    display:flex;
                    align-items:center;
                    gap:10px;
                    min-height:27px;
                    font-family:Arial,sans-serif;
                    font-size:13px;
                ">

                    <span style="
                        display:inline-block;
                        width:14px;
                        height:14px;
                        min-width:14px;
                        border-radius:50%;
                        background:{cor};
                        border:2px solid {cor_linha};
                    "></span>

                    <span style="
                        min-width:180px;
                        color:#e8eaed;
                        font-weight:{
                            'bold'
                            if estacao in caminho
                            else 'normal'
                        };
                    ">
                        {estacao}
                    </span>

                    <span style="
                        color:#bdc1c6;
                        font-size:11px;
                        font-weight:bold;
                    ">
                        {marca}
                    </span>

                </div>
                """
            )

        # ---------- LINHA VERTICAL ----------

        linhas_html.append(
            f"""
            <div style="
                border-left:4px solid {cor_linha};
                padding-left:12px;
                margin-left:7px;
                margin-bottom:22px;
            ">
                {''.join(estacoes_html)}
            </div>
            """
        )

    # ---------- CONTAINER PRINCIPAL ----------

    return """
    <div style="
        max-width:600px;
        padding:20px;
        background:#202124;
        color:#e8eaed;
        border:1px solid #3c4043;
        border-radius:10px;
    ">
    """ + "".join(linhas_html) + "</div>"

In [28]:
# ---------- PLANEJADOR ----------

TEMPO_POR_TRECHO = 2


def planejar(
    pedido,
    fechadas=(),
    manutencao=(),
    algoritmo="BFS",
    linhas_paralisadas=()
):
    """Planeja uma rota usando lógica + BFS/DFS."""

    # ---------- FATOS INICIAIS ----------

    fatos = fatos_base()

    tipo_o, nome_o = pedido["origem"]
    tipo_d, nome_d = pedido["destino"]


    # Origem

    fatos.add(
        ("usuario_esta_em", nome_o)
        if tipo_o == "local"
        else ("usuario_esta_na_estacao", nome_o)
    )


    # Destino

    fatos.add(
        ("usuario_quer_ir", nome_d)
        if tipo_d == "local"
        else ("usuario_quer_ir_estacao", nome_d)
    )


    # ---------- ACESSIBILIDADE ----------

    if pedido.get("acessibilidade"):
        fatos.add(
            ("precisa_acessibilidade",)
        )


    # ---------- ESTAÇÕES FECHADAS ----------

    for estacao in fechadas:
        fatos.add(
            ("fechada", estacao)
        )


    # ---------- ELEVADORES EM MANUTENÇÃO ----------

    for estacao in manutencao:
        fatos.add(
            ("elevador_em_manutencao", estacao)
        )


    # ---------- LINHAS PARALISADAS — R7 ----------

    for linha in linhas_paralisadas:
        fatos.add(
            ("linha_paralisada", linha)
        )


    # ---------- MOTOR DE INFERÊNCIA ----------

    fatos, justificativas = encadear_para_frente(
        fatos,
        REGRAS
    )


    # ---------- ORIGEM E DESTINO DEDUZIDOS ----------

    origem = consultar(
        fatos,
        "origem"
    )[0][0]

    destino = consultar(
        fatos,
        "destino"
    )[0][0]


    # ---------- ESTAÇÕES BLOQUEADAS ----------

    bloqueadas = {
        estacao
        for (estacao,) in consultar(
            fatos,
            "bloqueada"
        )
    }


    # ---------- ALERTAS ----------

    alertas = consultar(
        fatos,
        "alerta"
    )


    # ---------- ALGORITMO DE BUSCA ----------

    buscar = (
        bfs
        if algoritmo == "BFS"
        else dfs
    )


    caminho, visitados = buscar(
        GRAFO,
        origem,
        destino,
        bloqueadas
    )


    # ---------- BALDEAÇÕES ----------

    if caminho:

        qtd_baldeacoes, baldeacoes = contar_baldeacoes(
            caminho,
            LINHAS_DO_TRECHO
        )

    else:

        qtd_baldeacoes = 0
        baldeacoes = []


    # ---------- RESULTADO ----------

    return {

        "origem": origem,

        "destino": destino,

        "algoritmo": algoritmo,

        "caminho": caminho,

        "visitados": visitados,

        "bloqueadas": sorted(
            bloqueadas
        ),

        "alertas": [
            f"{papel}: {estacao}"
            for papel, estacao in alertas
        ],

        "paradas": (
            len(caminho) - 1
            if caminho
            else None
        ),

        "baldeacoes": baldeacoes,

        "qtd_baldeacoes": qtd_baldeacoes,

        "tempo_min": (
            (len(caminho) - 1)
            * TEMPO_POR_TRECHO
            if caminho
            else None
        ),

        "regras_usadas": sorted(
            set(
                justificativas.values()
            )
        ),
    }

In [29]:
# ---------- INTERFACE ----------

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


# ============================================================
# OPÇÕES
# ============================================================

opcoes = (
    [(f"📍 {local}", ("local", local)) for local in LOCAIS]
    +
    [(f"🚇 {estacao}", ("estacao", estacao)) for estacao in TODAS_ESTACOES]
)


# ============================================================
# WIDGETS
# ============================================================

txt_pedido = widgets.Textarea(
    placeholder=(
        "Ex.: Estou na Catedral da Sé e quero ir "
        "ao Terminal Rodoviário Jabaquara"
    ),
    layout=widgets.Layout(
        width="95%",
        height="70px"
    )
)


btn_interpretar = widgets.Button(
    description="Interpretar pedido",
    button_style="info"
)


dd_origem = widgets.Dropdown(
    options=opcoes,
    description="Origem:",
    layout=widgets.Layout(width="45%")
)


dd_destino = widgets.Dropdown(
    options=opcoes,
    value=("local", "Terminal Rodoviário Jabaquara"),
    description="Destino:",
    layout=widgets.Layout(width="45%")
)


chk_acess = widgets.Checkbox(
    description="Preciso de acessibilidade"
)


sel_fechadas = widgets.SelectMultiple(
    options=TODAS_ESTACOES,
    description="Fechadas:",
    rows=6,
    layout=widgets.Layout(width="45%")
)


sel_manut = widgets.SelectMultiple(
    options=TODAS_ESTACOES,
    description="Elevador:",
    rows=6,
    layout=widgets.Layout(width="45%")
)


rb_algoritmo = widgets.RadioButtons(
    options=["BFS", "DFS"],
    value="BFS",
    description="Busca:"
)


# ---------- R7 — LINHA PARALISADA ----------

sel_linhas_paralisadas = widgets.Dropdown(
    options=[
        ("Nenhuma", None),
        ("Linha 1-Azul", "Linha 1-Azul"),
        ("Linha 2-Verde", "Linha 2-Verde"),
        ("Linha 3-Vermelha", "Linha 3-Vermelha")
    ],
    value=None,
    description="Paralisada:",
    layout=widgets.Layout(width="45%")
)


btn_buscar = widgets.Button(
    description="Buscar rota",
    button_style="success"
)


# Duas áreas de saída diferentes
saida_interpretacao = widgets.Output()
saida_resultado = widgets.Output()


# ============================================================
# INTERPRETAR PEDIDO
# ============================================================

def ao_interpretar(_):

    with saida_interpretacao:

        clear_output()

        pedido, msg = interpretar_pedido(
            txt_pedido.value
        )

        print(msg)

        if pedido:

            dd_origem.value = pedido["origem"]
            dd_destino.value = pedido["destino"]
            chk_acess.value = pedido["acessibilidade"]

            print(
                "Campos preenchidos. "
                "Confira e clique em 'Buscar rota'."
            )


# ============================================================
# BUSCAR ROTA
# ============================================================

def ao_buscar(_):

    with saida_resultado:

        clear_output()

        # ---------- PEDIDO ----------

        pedido = {
            "origem": dd_origem.value,
            "destino": dd_destino.value,
            "acessibilidade": chk_acess.value
        }


        # ---------- LINHA PARALISADA ----------

        # O Dropdown devolve uma string.
        # O planejar() espera uma coleção de linhas.
        # Por isso transformamos:
        #
        # "Linha 1-Azul"
        #
        # em:
        #
        # ["Linha 1-Azul"]

        linhas_paralisadas = (
            [sel_linhas_paralisadas.value]
            if sel_linhas_paralisadas.value
            else []
        )


        # ====================================================
        # RESULTADO ESCOLHIDO PELO USUÁRIO
        # ====================================================

        r = planejar(
            pedido,
            fechadas=sel_fechadas.value,
            manutencao=sel_manut.value,
            algoritmo=rb_algoritmo.value,
            linhas_paralisadas=linhas_paralisadas
        )


        # ---------- TÍTULO ----------

        display(
            HTML(
                f"""
                <h3>
                    {r['origem']} → {r['destino']}
                </h3>
                """
            )
        )


        # ---------- NARRADOR ----------

        print(
            narrar(r)
        )

        print()


        # ====================================================
        # INFORMAÇÕES DA ROTA
        # ====================================================

        if r["caminho"]:

            print(
                f"Paradas: {r['paradas']}"
            )

            print(
                f"Baldeações: {r['qtd_baldeacoes']}"
            )


            # Mostra onde acontecem as baldeações

            if r["baldeacoes"]:

                for estacao, linha in r["baldeacoes"]:

                    print(
                        f"  • {estacao} → {linha}"
                    )


            print(
                f"Tempo estimado: "
                f"{r['tempo_min']} minutos"
            )


        else:

            print(
                "Nenhuma rota disponível."
            )


        print()


        # ====================================================
        # COMPARAÇÃO BFS x DFS
        # ====================================================

        resultado_bfs = planejar(
            pedido,
            fechadas=sel_fechadas.value,
            manutencao=sel_manut.value,
            algoritmo="BFS",
            linhas_paralisadas=linhas_paralisadas
        )


        resultado_dfs = planejar(
            pedido,
            fechadas=sel_fechadas.value,
            manutencao=sel_manut.value,
            algoritmo="DFS",
            linhas_paralisadas=linhas_paralisadas
        )


        display(
            HTML(
                """
                <h4>
                    Comparação dos algoritmos
                </h4>
                """
            )
        )


        print(
            f"BFS → "
            f"Paradas: {resultado_bfs['paradas']} | "
            f"Estações visitadas: "
            f"{len(resultado_bfs['visitados'])}"
        )


        print(
            f"DFS → "
            f"Paradas: {resultado_dfs['paradas']} | "
            f"Estações visitadas: "
            f"{len(resultado_dfs['visitados'])}"
        )


        print()


        # ====================================================
        # REGRAS LÓGICAS DISPARADAS
        # ====================================================

        print(
            "Regras disparadas: "
            + ", ".join(r["regras_usadas"])
        )


        # ====================================================
        # MAPA
        # ====================================================

        display(
            HTML(
                desenhar_linhas(r)
            )
        )


# ============================================================
# EVENTOS DOS BOTÕES
# ============================================================

btn_interpretar.on_click(
    ao_interpretar
)

btn_buscar.on_click(
    ao_buscar
)


# ============================================================
# PAINEL
# ============================================================

painel = widgets.VBox([

    # ---------- CABEÇALHO ----------

    widgets.HTML(
        """
        <div style="
            padding:15px;
            background:#202124;
            color:#e8eaed;
            border:1px solid #3c4043;
            border-radius:10px;
            margin-bottom:10px;
        ">

            <h2 style="
                margin:0;
                color:#e8eaed;
            ">
                MetrôBot SP 2.0
            </h2>

            <p style="
                margin:5px 0 0 0;
                color:#bdc1c6;
            ">
                Planejamento inteligente de rotas
                — Linhas 1, 2 e 3
            </p>

        </div>
        """
    ),


    # ---------- LINGUAGEM NATURAL ----------

    widgets.HTML(
        "<b>Pedido em linguagem natural</b>"
    ),

    txt_pedido,

    btn_interpretar,

    # A resposta do intérprete aparece aqui
    saida_interpretacao,


    widgets.HTML("<hr>"),


    # ---------- ORIGEM / DESTINO ----------

    widgets.HBox([
        dd_origem,
        dd_destino
    ]),


    # ---------- ACESSIBILIDADE / ALGORITMO ----------

    widgets.HBox([
        chk_acess,
        rb_algoritmo
    ]),


    # ---------- CONDIÇÕES DA REDE ----------

    widgets.HTML(
        "<b>Condições da rede</b>"
    ),

    widgets.HBox([
        sel_fechadas,
        sel_manut
    ]),


    # ---------- LINHA PARALISADA ----------

    sel_linhas_paralisadas,


    # ---------- BUSCAR ----------

    btn_buscar,


    # Resultado completo aparece aqui
    saida_resultado
])

In [30]:
# ---------- TESTES ----------

def rodar_testes():

    # =========================================================
    # TESTE 1 — Tucuruvi → Corinthians-Itaquera
    # Esperado: 22 paradas e 1 baldeação na Sé
    # =========================================================

    r = planejar({
        "origem": ("estacao", "Tucuruvi"),
        "destino": ("estacao", "Corinthians-Itaquera"),
        "acessibilidade": False
    })

    assert r["paradas"] == 22
    assert r["qtd_baldeacoes"] == 1
    assert r["baldeacoes"][0][0] == "Sé"


    # =========================================================
    # TESTE 2 — Vila Madalena → Jabaquara
    # Esperado: 14 paradas e 1 baldeação
    # em Paraíso ou Ana Rosa
    # =========================================================

    r = planejar({
        "origem": ("estacao", "Vila Madalena"),
        "destino": ("estacao", "Jabaquara"),
        "acessibilidade": False
    })

    assert r["paradas"] == 14
    assert r["qtd_baldeacoes"] == 1
    assert r["baldeacoes"][0][0] in ("Paraíso", "Ana Rosa")


    # =========================================================
    # TESTE 3 — Palmeiras-Barra Funda → Vila Prudente
    # Esperado: 16 paradas e 2 baldeações
    # =========================================================

    r = planejar({
        "origem": ("estacao", "Palmeiras-Barra Funda"),
        "destino": ("estacao", "Vila Prudente"),
        "acessibilidade": False
    })

    assert r["paradas"] == 16
    assert r["qtd_baldeacoes"] == 2

    estacoes_baldeacao = [
        estacao for estacao, _linha in r["baldeacoes"]
    ]

    assert "Sé" in estacoes_baldeacao
    assert (
        "Paraíso" in estacoes_baldeacao
        or "Ana Rosa" in estacoes_baldeacao
    )


    # =========================================================
    # TESTE 4 — Tucuruvi → Brás com Sé fechada
    # Esperado: sem rota
    # =========================================================

    r = planejar({
        "origem": ("estacao", "Tucuruvi"),
        "destino": ("estacao", "Brás"),
        "acessibilidade": False
    }, fechadas=["Sé"])

    assert r["caminho"] is None


    # =========================================================
    # TESTE 5 — Vila Madalena → Jabaquara com Paraíso fechada
    # Esperado: sem rota
    # =========================================================

    r = planejar({
        "origem": ("estacao", "Vila Madalena"),
        "destino": ("estacao", "Jabaquara"),
        "acessibilidade": False
    }, fechadas=["Paraíso"])

    assert r["caminho"] is None


    # =========================================================
    # TESTE 6 — Vila Prudente → Jabaquara com Paraíso fechada
    # Esperado: 13 paradas e caminho passando por Ana Rosa
    # =========================================================

    r = planejar({
        "origem": ("estacao", "Vila Prudente"),
        "destino": ("estacao", "Jabaquara"),
        "acessibilidade": False
    }, fechadas=["Paraíso"])

    assert r["caminho"] is not None
    assert r["paradas"] == 13
    assert "Ana Rosa" in r["caminho"]


    # =========================================================
    # TESTE 7 — R6 deve deduzir as integrações automaticamente
    # =========================================================

    fatos, _ = encadear_para_frente(
        fatos_base(),
        REGRAS
    )

    integracoes = {
        e for (e,) in consultar(fatos, "integracao")
    }

    assert integracoes == {
        "Sé",
        "Paraíso",
        "Ana Rosa"
    }


    # =========================================================
    # TESTE 8 — R7: linha paralisada
    # Uma estação pertencente à linha deve ser bloqueada
    # =========================================================

    fatos = fatos_base()
    fatos.add(
        ("linha_paralisada", "Linha 3-Vermelha")
    )

    fatos, _ = encadear_para_frente(
        fatos,
        REGRAS
    )

    bloqueadas = {
        e for (e,) in consultar(fatos, "bloqueada")
    }

    assert "Corinthians-Itaquera" in bloqueadas


    print("Todos os testes passaram!")


rodar_testes()

Todos os testes passaram!


In [31]:
display(painel)